<a href="https://colab.research.google.com/github/minbot616/Agentic-AI-Lab/blob/main/agentic_lab_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-community langchain-groq chromadb \
sentence-transformers pypdf langchain_text_splitters langchain_huggingface -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 110.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/

In [ ]:
from langchain_groq import ChatGroq

# IMPORTANT: Replace "PASTE_YOUR_GROQ_KEY_HERE" with your actual Groq API key.
# You can obtain an API key from https://console.groq.com/keys
llm = ChatGroq(api_key="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",
               model="llama-3.1-8b-instant", temperature=0)

In [ ]:
def answer_from_pdf(query):

    docs = db.as_retriever().invoke(query)

    context = '\n'.join(d.page_content for d in docs)

    prompt = (f"Use ONLY this context to answer.\n{context}\n\n"

              f"Question: {query}\n"

              "If the context does not contain the answer, reply exactly: "

              "'I don't know'.")

    return llm.invoke(prompt).content

In [ ]:
# Download a sample PDF file (you can replace this with your own PDF)
!wget -O sample.pdf https://www.orimi.com/pdf-test.pdf

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Placeholder for PDF path. Replace with your actual PDF file.
pdf_path = "sample.pdf" # Ensure you upload an 'example.pdf' file to your Colab environment

# Load PDF
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# Split documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(documents)

# Create embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create vector store
db = Chroma.from_documents(docs, embeddings)

def adaptive_answer(question, max_tries=3):

    query = question

    for attempt in range(1, max_tries + 1):

        print(f'Attempt {attempt} with query: {query}')

        answer = answer_from_pdf(query)

        if "i don't know" not in answer.lower():

            return answer          # good answer, stop

        # otherwise, rephrase and retry

        query = llm.invoke(

            f'Rephrase this search query differently: {query}').content

    return 'Could not find an answer after several tries.'



print(adaptive_answer('What is the main conclusion of the document?'))

--2026-07-17 05:07:29--  https://www.orimi.com/pdf-test.pdf
Resolving www.orimi.com (www.orimi.com)... 109.120.172.1
Connecting to www.orimi.com (www.orimi.com)|109.120.172.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 20597 (20K) [application/pdf]
Saving to: ‘sample.pdf’

sample.pdf          100%[===================>]  20.11K  --.-KB/s    in 0s      

2026-07-17 05:07:30 (137 MB/s) - ‘sample.pdf’ saved [20597/20597]



/tmp/ipykernel_600/3022198831.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Attempt 1 with query: What is the main conclusion of the document?
The main conclusion of the document is that the computer is equipped with a PDF reader, allowing the user to view PDF documents and forms available on the site.
